# AI-RAG · Week 17 — Notebook 1 (Day 1)
## Planning & Reasoning — Planner-Executor, Reflection, and *When to Plan*

The **danger zone**: three big ideas stack this week (planning, streaming, HITL). Day 1 is planning & reasoning — the **planner-executor** pattern, **reflection loops**, the **meta-decision** (plan or not?), and Tree-of-Thought (named only). Runs **fully offline** on a mock LLM. Builds on the W16 LangGraph agent.

> New LangGraph code, executed. The tools + mock are carried from W13–W16.

### Setup

In [1]:
# --- notebook setup ----------------------------------------------------------
# Installs LangGraph (pinned) + pydantic, writes fixtures, builds the side-DB.
# Runs fully offline on a deterministic mock LLM — no OpenAI/Tavily key, no cost.
import subprocess, sys
for pkg in ("langgraph>=1.0,<2","pydantic>=2.5"):
    mod = pkg.split(">")[0].split("=")[0].replace("-","_")
    try: __import__(mod)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])

from pathlib import Path
Path("data").mkdir(exist_ok=True)
Path("data/web_fixtures.json").write_text(r"""{
  "ai agents": [
    {"title": "State of AI Agents 2026", "snippet": "Agentic systems moved from demos to production in 2026, with tool-use reliability the key differentiator."},
    {"title": "Agent frameworks roundup", "snippet": "Raw loops and graph-based orchestration remain the two dominant patterns."}
  ],
  "openai api": [
    {"title": "OpenAI API pricing", "snippet": "gpt-4o-mini is positioned as the low-cost tier for high-volume workloads."}
  ],
  "cricket": [
    {"title": "Cricket World Cup — result", "snippet": "The most recent tournament concluded with a closely contested final."}
  ],
  "python libraries": [
    {"title": "Trending Python libraries", "snippet": "Data, LLM-tooling, and async libraries dominate this year's trending lists."}
  ],
  "announce": [
    {"title": "OpenAI announcements", "snippet": "Recent announcements focus on tool use, structured outputs, and cost reductions."}
  ],
  "_default": [
    {"title": "Web result", "snippet": "A relevant public-web snippet would appear here (offline fixture)."}
  ]
}""" + "\n")
import langgraph; print("langgraph", getattr(langgraph,"__version__","?"), "· offline")


langgraph ? · offline


### The carried tools + mock LLM

The tools + mock LLM (carried from W13–W16). This week we add planning, reflection, streaming, and HITL on top.

In [2]:
"""src/agents/tools/base.py — the Tool contract.

A tool an LLM can call reliably is four things done well (the W13 principles):
  1. a good NAME       — a verb, specific, snake_case
  2. a typed SCHEMA    — a Pydantic model: required fields, sensible defaults
  3. a DESCRIPTION     — written for the *model*, saying WHEN to use the tool
  4. IDEMPOTENCY        — declared, so the loop knows if a retry is safe

Every tool subclasses `Tool`, sets those four, and implements `run`. The same
definition exports to both OpenAI function-calling and Anthropic tool-use schemas.
"""
from __future__ import annotations

from typing import Any

from pydantic import BaseModel


class Tool:
    name: str = ""
    description: str = ""
    Args: type[BaseModel] = BaseModel
    idempotent: bool = True          # is calling this twice with the same args safe?
    has_side_effects: bool = False   # does it change external state?

    def run(self, **kwargs: Any) -> str:
        raise NotImplementedError

    # ── validation + execution ────────────────────────────────────────────
    def call(self, arguments: dict[str, Any]) -> str:
        """Validate arguments against the schema, then run."""
        args = self.Args(**(arguments or {}))
        return self.run(**args.model_dump())

    # ── provider schema exports ───────────────────────────────────────────
    def parameters_schema(self) -> dict:
        return self.Args.model_json_schema()

    def openai_schema(self) -> dict:
        return {"type": "function",
                "function": {"name": self.name,
                             "description": self.description,
                             "parameters": self.parameters_schema()}}

    def anthropic_schema(self) -> dict:
        return {"name": self.name,
                "description": self.description,
                "input_schema": self.parameters_schema()}


In [3]:
"""src/agents/tools/calculate.py — a computation tool.

Deterministic, offline, and safe: it evaluates an arithmetic expression with a
small AST walker (no `eval`), so an LLM can hand it a calculation and get a
reliable number back. Idempotent and side-effect-free — the easy end of the
tool spectrum, and a good first example of a well-formed tool.
"""
from __future__ import annotations

import ast
import operator

from pydantic import BaseModel, Field

# Tool defined above

_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.Mod: operator.mod, ast.Pow: operator.pow,
    ast.USub: operator.neg, ast.UAdd: operator.pos, ast.FloorDiv: operator.floordiv,
}


def _eval(node: ast.AST) -> float:
    if isinstance(node, ast.Expression):
        return _eval(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval(node.left), _eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval(node.operand))
    raise ValueError("unsupported expression — only + - * / % ** ( ) and numbers")


class CalculateArgs(BaseModel):
    expression: str = Field(
        ..., description="An arithmetic expression, e.g. '1350 * 0.18' or '(42 + 8) / 2'.")


class CalculateTool(Tool):
    name = "calculate"
    description = ("Evaluate an arithmetic expression and return the numeric result. "
                   "Use this whenever a question needs exact math — sums, differences, "
                   "products, division, percentages, or powers. Do NOT use it to look up "
                   "facts; it only does arithmetic on numbers you provide.")
    Args = CalculateArgs
    idempotent = True
    has_side_effects = False

    def run(self, expression: str) -> str:
        try:
            result = _eval(ast.parse(expression, mode="eval"))
        except Exception as e:  # noqa: BLE001 - report cleanly to the agent
            return f"calculate error: {e}"
        if isinstance(result, float) and result.is_integer():
            result = int(result)
        return f"{expression} = {result}"


In [4]:
"""src/agents/tools/query_sql.py — SQL as a tool, safely.

Letting an LLM run SQL is powerful and dangerous. This tool ships the three rails
every SQL-as-a-tool needs in production:

  1. AUDIT LOGGING     — every query (allowed or denied) is written to audit_log
  2. READ-ONLY DEFAULT  — only a single SELECT / WITH…SELECT is permitted
  3. CONSTRAINED UPDATES — writes/DDL (INSERT/UPDATE/DELETE/DROP/…) are denied
                          outright (write access is a W18 topic, gated behind
                          injection defenses and a human-approval step)

Everything runs against a local SQLite database, so it works offline with no
credentials. The migration path to Postgres is a config change, not a rewrite
(see docs/runbooks/sql_as_a_tool_safety.md).
"""
from __future__ import annotations

import re
import sqlite3
from datetime import datetime, timezone
from pathlib import Path

from pydantic import BaseModel, Field

# Tool defined above

DB_PATH = Path("data/capstone_side.db")

# Statements that change data or schema — denied by the read-only rail.
_FORBIDDEN = {"insert", "update", "delete", "drop", "alter", "create", "replace",
              "truncate", "attach", "detach", "pragma", "vacuum", "reindex",
              "grant", "revoke", "begin", "commit"}
_MAX_ROWS = 50


def _strip_comments(sql: str) -> str:
    sql = re.sub(r"--[^\n]*", " ", sql)
    sql = re.sub(r"/\*.*?\*/", " ", sql, flags=re.S)
    return sql.strip()


def is_read_only(sql: str) -> tuple[bool, str]:
    """Return (ok, reason). Only a single SELECT/WITH…SELECT is allowed."""
    clean = _strip_comments(sql)
    if not clean:
        return False, "empty query"
    statements = [s for s in clean.split(";") if s.strip()]
    if len(statements) > 1:
        return False, "multiple statements are not allowed"
    first = statements[0].lstrip().split(None, 1)[0].lower()
    if first not in ("select", "with"):
        return False, f"only SELECT queries are allowed (got '{first.upper()}')"
    words = set(re.findall(r"[a-z_]+", clean.lower()))
    hit = words & _FORBIDDEN
    if hit:
        return False, f"write/DDL keyword not permitted: {', '.join(sorted(hit))}"
    return True, "ok"


class QuerySqlArgs(BaseModel):
    sql: str = Field(..., description="A single read-only SQL SELECT query over the "
                                      "document_metadata table.")
    purpose: str = Field("", description="One phrase describing what the query is for "
                                         "(recorded in the audit log).")


class QuerySqlTool(Tool):
    name = "query_sql"
    description = (
        "Run a read-only SQL SELECT against the knowledge-base metadata database and "
        "return the rows. Use this for questions about the DOCUMENTS THEMSELVES — how "
        "many documents exist, which department owns them, their type, version, "
        "chunk_count, or when they were last updated. Table: document_metadata"
        "(id, source, title, doc_type, department, version, updated_at, chunk_count). "
        "Do NOT use it to answer questions about document CONTENTS — only their metadata.")
    Args = QuerySqlArgs
    idempotent = True            # SELECTs don't change state
    has_side_effects = False

    def __init__(self, db_path: Path | str = DB_PATH):
        self.db_path = Path(db_path)

    def _audit(self, request: str, sql: str, decision: str,
               row_count: int | None, error: str | None) -> None:
        con = sqlite3.connect(self.db_path)
        con.execute(
            "INSERT INTO audit_log (ts, actor, request, sql, decision, row_count, error) "
            "VALUES (?,?,?,?,?,?,?)",
            (datetime.now(timezone.utc).isoformat(timespec="seconds"), "agent",
             request, sql, decision, row_count, error))
        con.commit()
        con.close()

    def run(self, sql: str, purpose: str = "") -> str:
        ok, reason = is_read_only(sql)
        if not ok:
            self._audit(purpose, sql, "denied", None, reason)
            return (f"query_sql DENIED: {reason}. This tool is read-only; write access "
                    f"is not available. (Logged to the audit trail.)")
        try:
            con = sqlite3.connect(self.db_path)
            cur = con.execute(sql)
            rows = cur.fetchmany(_MAX_ROWS)
            cols = [d[0] for d in cur.description] if cur.description else []
            con.close()
        except Exception as e:  # noqa: BLE001
            self._audit(purpose, sql, "error", None, str(e))
            return f"query_sql error: {e}"
        self._audit(purpose, sql, "allowed", len(rows), None)
        if not rows:
            return "query_sql: (0 rows)"
        header = " | ".join(cols)
        body = "\n".join(" | ".join(str(v) for v in r) for r in rows)
        more = "" if len(rows) < _MAX_ROWS else f"\n… (capped at {_MAX_ROWS} rows)"
        return f"{header}\n{'-' * len(header)}\n{body}{more}"


In [5]:
"""src/agents/tools/search_web.py — an external-API tool (web search).

Wraps a web-search API (Tavily) so the agent can reach information beyond the
knowledge base — current events, prices, anything not in the corpus. This is the
"API-calling agent" pattern: the LLM decides it needs fresh external info and
calls out.

Web search has real cost and latency, so the description tells the model to use
it only when the answer isn't in the corpus. For offline runs and tests, the tool
falls back to a local fixture file, so the whole thing works with no network and
no TAVILY_API_KEY.
"""
from __future__ import annotations

import json
import os
from pathlib import Path

from pydantic import BaseModel, Field

# Tool defined above

FIXTURES_PATH = Path("data/web_fixtures.json")


class SearchWebArgs(BaseModel):
    query: str = Field(..., description="A concise web-search query.")
    max_results: int = Field(3, ge=1, le=5, description="How many results to return.")


class SearchWebTool(Tool):
    name = "search_web"
    description = (
        "Search the public web and return short result snippets. Use this ONLY when the "
        "answer is not in the internal knowledge base — for current events, live figures, "
        "recent releases, or anything outside company documents. It has cost and latency, "
        "so prefer the internal tools when they can answer. Returns titles and snippets, "
        "not full pages.")
    Args = SearchWebArgs
    idempotent = True            # reading search results doesn't change state
    has_side_effects = False

    def __init__(self, offline: bool | None = None):
        # offline when explicitly set, or when there is no API key available
        self.offline = (not os.environ.get("TAVILY_API_KEY")) if offline is None else offline

    def _fixture(self, query: str, k: int) -> str:
        try:
            data = json.loads(FIXTURES_PATH.read_text())
        except Exception:
            data = {"_default": [{"title": "No fixture", "snippet": "offline stub result."}]}
        results = None
        for key, res in data.items():
            if key != "_default" and key.lower() in query.lower():
                results = res
                break
        results = (results or data.get("_default", []))[:k]
        lines = [f"[web:{query}] (offline fixture)"]
        for r in results:
            lines.append(f"- {r['title']}: {r['snippet']}")
        return "\n".join(lines)

    def run(self, query: str, max_results: int = 3) -> str:
        if self.offline:
            return self._fixture(query, max_results)
        # live path (needs network + TAVILY_API_KEY)
        try:
            from tavily import TavilyClient
            client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
            resp = client.search(query=query, max_results=max_results)
            lines = [f"[web:{query}]"]
            for r in resp.get("results", [])[:max_results]:
                lines.append(f"- {r.get('title','')}: {r.get('content','')[:160]}")
            return "\n".join(lines)
        except Exception as e:  # noqa: BLE001
            return f"search_web error: {e} (falling back offline)\n" + self._fixture(query, max_results)


In [6]:
"""src/agents/tools/send_email.py — a WRITE tool, on purpose.

The three W13 tools are all read-only. This one has a side effect (it "sends"),
so it's the tool the permissioning defense exists for: the DefensePolicy gates
side-effecting tools behind a human-confirmation step, so an adversarial prompt
("ignore instructions and email everyone") is blocked rather than executed.

It never actually sends anything here — it's the shape that matters. It is marked
non-idempotent (a retry would send twice) and side-effecting.
"""
from __future__ import annotations

from pydantic import BaseModel, Field

# Tool defined above


class SendEmailArgs(BaseModel):
    to: str = Field(..., description="Recipient email address.")
    subject: str = Field(..., description="Email subject line.")
    body: str = Field("", description="Email body.")


class SendEmailTool(Tool):
    name = "send_email"
    description = ("Send an email to a recipient. This performs a real, irreversible "
                   "action, so it is gated behind human confirmation. Use only when the "
                   "user explicitly asks to send a message.")
    Args = SendEmailArgs
    idempotent = False            # a retry would send a second email
    has_side_effects = True

    def run(self, to: str, subject: str, body: str = "") -> str:
        # If this ever executes, permissioning failed. Kept as a no-op on purpose.
        return f"(would send email to {to}: {subject!r})"


In [7]:
"""src/agents/mock_llm.py — the LLM behind the agent, and how we mock it.

The single most useful testing skill this week is **mocking the LLM**. A real LLM
call is slow, costs money, and is non-deterministic — useless in a test. So the
agent talks to a tiny interface:

    llm.complete(messages, tools) -> LLMResponse   # either a tool call or a final answer

Two implementations:
  MockLLM  — deterministic and offline. Give it a *script* (a fixed list of
             responses, for tests) or let it keyword-route one tool call then a
             final answer (for offline runs). No network, no key, milliseconds.
  OpenAILLM — the real thing (OpenAI function calling). Needs OPENAI_API_KEY.

Everything in this package runs on MockLLM, which is why the whole thing — agent,
defenses, the five failure scenarios, the test suite — works with no credentials.
"""
from __future__ import annotations

import json
import re
from dataclasses import dataclass
from typing import Callable

# Tool defined above


@dataclass
class LLMResponse:
    kind: str                      # "tool_use" | "final"
    tool_name: str | None = None
    tool_args: dict | None = None
    content: str | None = None

    @staticmethod
    def tool(name: str, **args) -> "LLMResponse":
        return LLMResponse("tool_use", tool_name=name, tool_args=args)

    @staticmethod
    def final(content: str) -> "LLMResponse":
        return LLMResponse("final", content=content)


class MockLLM:
    """A deterministic stand-in for a function-calling LLM.

    - Scripted:  MockLLM(script=[LLMResponse.tool('calculate', expression='2+2'),
                                 LLMResponse.final('It is 4.')])
                 returns those in order (extra calls repeat the last one).
    - Keyword:   MockLLM()  → routes ONE tool call by keyword, then a final answer
                 synthesised from the observation. Good for offline demos.
    """

    def __init__(self, script: list[LLMResponse] | None = None,
                 router: Callable[[list[dict], list[Tool]], LLMResponse] | None = None):
        self.script = list(script) if script else None
        self.router = router
        self.calls = 0

    def complete(self, messages: list[dict], tools: list[Tool]) -> LLMResponse:
        self.calls += 1
        if self.script is not None:
            return self.script.pop(0) if self.script else LLMResponse.final(
                "(no more scripted responses)")
        if self.router is not None:
            return self.router(messages, tools)
        return self._keyword_route(messages, tools)

    # ── default keyword routing: one tool call, then a final answer ──────────
    def _keyword_route(self, messages: list[dict], tools: list[Tool]) -> LLMResponse:
        # If a tool has already produced an observation, answer from it.
        obs = _last_observation(messages)
        if obs is not None:
            return LLMResponse.final(f"Based on the tool result: {obs}")
        q = _last_user(messages).lower()
        names = {t.name for t in tools}
        if "calculate" in names and re.search(r"\d\s*[-+*/%]\s*\d|percent|%|how much is", q):
            return LLMResponse.tool("calculate", expression=_extract_expr(q))
        sql_kw = ("how many doc", "which department", "what version", "list the", "count",
                  "how many hr", "how many it", "how many finance", "updated", "chunk")
        if "query_sql" in names and any(k in q for k in sql_kw):
            return LLMResponse.tool("query_sql",
                                    sql="SELECT COUNT(*) AS total_documents FROM document_metadata",
                                    purpose="metadata question")
        web_kw = ("latest", "current", "today", "news", "price", "recent", "2026", "who won")
        if "search_web" in names and any(k in q for k in web_kw):
            return LLMResponse.tool("search_web", query=_last_user(messages), max_results=3)
        return LLMResponse.final("I can answer that directly: (no tool needed).")


# ─── real OpenAI adapter (needs a key + network) ────────────────────────────
class OpenAILLM:
    def __init__(self, model: str = "gpt-5-mini"):
        from openai import OpenAI
        self.client = OpenAI()
        self.model = model

    def complete(self, messages: list[dict], tools: list[Tool]) -> LLMResponse:
        resp = self.client.chat.completions.create(
            model=self.model, messages=messages,
            tools=[t.openai_schema() for t in tools], tool_choice="auto")
        msg = resp.choices[0].message
        if msg.tool_calls:
            call = msg.tool_calls[0]
            return LLMResponse.tool(call.function.name, **json.loads(call.function.arguments or "{}"))
        return LLMResponse.final(msg.content or "")


# ─── tiny helpers ───────────────────────────────────────────────────────────
def _last_user(messages: list[dict]) -> str:
    for m in reversed(messages):
        if m.get("role") == "user":
            return str(m.get("content", ""))
    return ""


def _last_observation(messages: list[dict]) -> str | None:
    for m in reversed(messages):
        if m.get("role") == "tool":
            return str(m.get("content", ""))
    return None


def _extract_expr(q: str) -> str:
    m = re.search(r"[0-9][0-9\.\s\+\-\*/%\(\)]*[0-9\)]", q)
    return m.group(0).strip() if m else "0"


### Build the side-database

In [8]:
#!/usr/bin/env python3
"""scripts/seed_db.py — (re)create the capstone side database.

W13 gives the assistant structured data to query as a tool. This builds a small
SQLite database with two tables:

  document_metadata  — one row per document in the knowledge base (the structured
                       side-table the SQL tool queries)
  audit_log          — every SQL query the agent runs is logged here (the audit
                       rail for SQL-as-a-tool)

SQLite is local and offline, so the whole SQL-tool exercise runs with no network
and no credentials. Run this once (smoke_test.sh does it for you):

    python3 scripts/seed_db.py
"""
from __future__ import annotations

import sqlite3
from pathlib import Path

DB_PATH = Path("data/capstone_side.db")  #  / "data" / "capstone_side.db"

# A small, realistic slice of the Enterprise Knowledge Assistant's corpus,
# across the three departments the capstone covers.
DOCS = [
    # source, title, doc_type, department, version, updated_at, chunk_count
    ("hr-001", "Leave & Time-Off Policy",          "policy",   "HR",      "v3", "2026-01-14", 42),
    ("hr-002", "Code of Conduct",                  "policy",   "HR",      "v2", "2025-11-02", 55),
    ("hr-003", "Remote Work Guidelines",           "guideline","HR",      "v4", "2026-02-20", 28),
    ("hr-004", "Onboarding Handbook",              "handbook", "HR",      "v1", "2025-06-30", 90),
    ("it-001", "Acceptable Use Policy",            "policy",   "IT",      "v5", "2026-01-30", 37),
    ("it-002", "Security Incident Runbook",        "runbook",  "IT",      "v2", "2026-02-11", 33),
    ("it-003", "VPN & Access Setup",               "guideline","IT",      "v3", "2025-12-05", 19),
    ("it-004", "Data Classification Standard",     "standard", "IT",      "v2", "2026-01-08", 24),
    ("fin-001","Expense Reimbursement Policy",     "policy",   "Finance", "v3", "2026-01-22", 31),
    ("fin-002","Procurement Guidelines",           "guideline","Finance", "v1", "2025-09-15", 40),
    ("fin-003","Travel Policy",                    "policy",   "Finance", "v2", "2025-10-19", 26),
    ("fin-004","Corporate Card Terms",             "terms",    "Finance", "v4", "2026-02-01", 18),
]


def seed(db_path: Path = DB_PATH) -> Path:
    db_path.parent.mkdir(parents=True, exist_ok=True)
    if db_path.exists():
        db_path.unlink()
    con = sqlite3.connect(db_path)
    cur = con.cursor()
    cur.execute("""
        CREATE TABLE document_metadata (
            id           INTEGER PRIMARY KEY,
            source       TEXT NOT NULL,
            title        TEXT NOT NULL,
            doc_type     TEXT NOT NULL,
            department   TEXT NOT NULL,
            version      TEXT NOT NULL,
            updated_at   TEXT NOT NULL,
            chunk_count  INTEGER NOT NULL
        )""")
    cur.executemany(
        "INSERT INTO document_metadata "
        "(source,title,doc_type,department,version,updated_at,chunk_count) "
        "VALUES (?,?,?,?,?,?,?)", DOCS)
    cur.execute("""
        CREATE TABLE audit_log (
            id         INTEGER PRIMARY KEY AUTOINCREMENT,
            ts         TEXT NOT NULL,
            actor      TEXT NOT NULL,
            request    TEXT,
            sql        TEXT,
            decision   TEXT NOT NULL,
            row_count  INTEGER,
            error      TEXT
        )""")
    con.commit()
    con.close()
    return db_path

seed()
print('side-DB built:', DB_PATH)


side-DB built: data/capstone_side.db


### A planner/reflection-aware mock

Our mock LLM plans (on a `PLAN` prompt), reflects (on a `REFLECT` prompt), or answers — deterministically, so the graph is reproducible offline.

In [9]:
import json
def w17_router(messages, tools):
    """A mock LLM that plans (PLAN prompt), reflects (REFLECT prompt), or answers."""
    system = next((m["content"] for m in messages if m["role"]=="system"), "")
    user   = next((m["content"] for m in messages if m["role"]=="user"), "")
    if system.startswith("PLAN"):
        q = user.lower(); steps=[]
        if any(w in q for w in ["how many","count","documents","compare"]):
            steps.append({"tool":"query_sql","args":{"sql":"SELECT department, COUNT(*) FROM document_metadata GROUP BY department","purpose":"count"}})
        if any(w in q for w in ["total","calculate"]):
            steps.append({"tool":"calculate","args":{"expression":"12 * 2"}})
        return LLMResponse.final(json.dumps(steps or [{"tool":"query_sql","args":{"sql":"SELECT COUNT(*) FROM document_metadata","purpose":"count"}}]))
    if system.startswith("REFLECT"):
        last = messages[-1]["content"].lower()
        return LLMResponse.final("redo" if ("error" in last or "no results" in last) else "ok")
    return LLMResponse.final("done: " + user[:40])

llm = MockLLM(router=w17_router)


## 1 · The planner-executor pattern

For a **multi-step** query, one LLM call writes a **plan** (a list of steps); an executor then runs each step. Useful when step *ordering* matters. We build it as a LangGraph: `plan → execute ⇄ reflect → final`.

In [10]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END

TOOLS = {t.name: t for t in [SearchWebTool(offline=True), QuerySqlTool(), CalculateTool(), SendEmailTool()]}

class PlanState(TypedDict):
    question: str
    plan: list
    step: int
    obs: Annotated[list, lambda a, b: a + b]
    reflections: int
    answer: str

def plan_node(s):
    r = llm.complete([{'role':'system','content':'PLAN the steps'}, {'role':'user','content':s['question']}], [])
    return {'plan': json.loads(r.content), 'step': 0}

def execute_node(s):
    step = s['plan'][s['step']]
    obs = TOOLS[step['tool']].call(step['args'])
    return {'obs': [f"{step['tool']}: {str(obs)[:60]}"], 'step': s['step'] + 1}

def route(s):
    return 'execute' if s['step'] < len(s['plan']) else 'final'

def final_node(s):
    return {'answer': f"completed {len(s['plan'])} steps"}

*(we add the reflection node in §2; first, the bare planner-executor)*

In [11]:
g = StateGraph(PlanState)
g.add_node('plan', plan_node); g.add_node('execute', execute_node); g.add_node('final', final_node)
g.add_edge(START, 'plan'); g.add_edge('plan', 'execute')
g.add_conditional_edges('execute', route, {'execute': 'execute', 'final': 'final'})
g.add_edge('final', END)
planner = g.compile()
out = planner.invoke({'question': 'how many documents are there and calculate the total',
                      'plan': [], 'step': 0, 'obs': [], 'reflections': 0, 'answer': ''})
print('plan:', [s['tool'] for s in out['plan']])
print('observations:', out['obs'])
print('answer:', out['answer'])

plan: ['query_sql', 'calculate']
observations: ['query_sql: department | COUNT(*)\n---------------------\nFinance | 4\nHR |', 'calculate: 12 * 2 = 24']
answer: completed 2 steps


## 2 · Reflection loops + self-correction

After a tool returns, a **reflection** node asks *'does this output make sense?'* before continuing — catching W14's **silent** failures (a valid-looking but wrong result). The cost is real: **reflection is +1 LLM call per step**, so use it where silent failures are expensive. Add a `reflect` node after `execute`:

In [12]:
def reflect_node(s):
    r = llm.complete([{'role':'system','content':'REFLECT on the output'}, {'role':'user','content':s['obs'][-1]}], [])
    # 'redo' would route back to re-plan; here we just count (all outputs are 'ok' offline)
    return {'reflections': s['reflections'] + 1}

g = StateGraph(PlanState)
g.add_node('plan', plan_node); g.add_node('execute', execute_node)
g.add_node('reflect', reflect_node); g.add_node('final', final_node)
g.add_edge(START, 'plan'); g.add_edge('plan', 'execute'); g.add_edge('execute', 'reflect')
g.add_conditional_edges('reflect', route, {'execute': 'execute', 'final': 'final'})
g.add_edge('final', END)
planner_r = g.compile()
out = planner_r.invoke({'question': 'how many documents and calculate the total',
                        'plan': [], 'step': 0, 'obs': [], 'reflections': 0, 'answer': ''})
print('steps:', len(out['plan']), '| reflections (=+1 LLM call each):', out['reflections'])
print(planner_r.get_graph().draw_mermaid())

steps: 2 | reflections (=+1 LLM call each): 2
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	plan(plan)
	execute(execute)
	reflect(reflect)
	final(final)
	__end__([<p>__end__</p>]):::last
	__start__ --> plan;
	execute --> reflect;
	plan --> execute;
	reflect -.-> execute;
	reflect -.-> final;
	final --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 3 · When planning helps vs adds latency (the meta-decision)

The counterintuitive production lesson: **planning ADDS latency** (an extra LLM call before any work). Single-step questions shouldn't go through a planner — *'what's the weather'* doesn't need a plan. Rough rule: **plan when ≥3 tool calls expected; skip when ≤2**. A cheap **meta-decision** (a heuristic, not another LLM call) chooses the path:

In [13]:
def needs_planning(question: str) -> bool:
    q = question.lower()
    hits = sum(k in q for k in ['how many','count','compare','across','total','calculate','and'])
    return hits >= 2   # heuristic: multi-step signals → plan; else skip

for q in ['what is the leave policy?',
          'how many documents are there and calculate the total across departments']:
    print(f"{'PLAN ' if needs_planning(q) else 'SKIP '} · {q}")

SKIP  · what is the leave policy?
PLAN  · how many documents are there and calculate the total across departments


**Trade-off, honestly:** measure whether planning improved *quality* or just added latency. Most single-step queries are faster and just as good without it — and the meta-decision itself, if you make it an LLM call, adds latency too (a heuristic is cheaper).

## 4 · Tree-of-Thought — named only

**Planner-executor** commits to *one* linear plan. **Tree-of-Thought (ToT)** instead explores *multiple* reasoning branches and picks the best — more powerful for genuinely open problems, but **many more LLM calls** (branch width × depth) and slower.

> **Decision rule:** planner-executor for **known-structure** tasks; ToT only when you **can't commit** to one plan upfront. Production usually doesn't need ToT — planner-executor + reflection covers the large majority of production cases. (ToT is the lab Stretch; not built here.)

## Recap

You built a **planner-executor** (plan → execute ⇄ reflect → final), added **reflection** (+1 call/step, catches silent failures), and a **meta-decision** that skips planning for single-step questions. ToT is the named alternative. Next, **Notebook 2**: streaming the agent's work to a UI and pausing for **human approval** on risky actions.